In [ ]:
# Import libraries
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import numpy as np
import pickle

from utils.graph import create_graph, get_icd9_info_for_Gu_mimic, draw_graph

In [ ]:
timeseries_dir= r"data/MIMICIII_last48h_ts2h/timeseries"
train_dir = r"data/MIMICIII_last48h_ts2h/train"
Path(train_dir).mkdir(exist_ok=True, parents=True)

## Build Usability Graph

In the usability graph G_u = (N_u, E_u, r_u), the nodes are icd9 codes and the edges are the usability between icd9 codes. The weight of the edge represents the usability between two icd9 codes.

The usability is defined as the similarity between records of two icd9 codes. The similarity is calculated based on the mean, std, and miss rate of the records of each icd9 code.

In [ ]:
# load data
demo_df = pd.read_csv(os.path.join(train_dir, "demographics.csv"), sep=',', dtype={'icd9_code': str})
ts_df = pd.read_csv(os.path.join(timeseries_dir, "time-series.csv"), sep=',')

icd9_codes = demo_df.icd9_code.unique()

In [ ]:
# create graph
G_u = create_graph([str(icd9_code) for icd9_code in icd9_codes])
difference_dict = {}

# 获取所有节点的 ICD9 信息
icd9_info_dict = get_icd9_info_for_Gu_mimic(demo_df, ts_df, icd9_codes,)

all_differences = []

# 遍历每个节点对，计算差异并存储
for i, node1 in enumerate(icd9_codes):
    for j, node2 in enumerate(icd9_codes[i+1:], i+1):  # 只计算一次 (node1, node2)，避免重复
        data_info1 = icd9_info_dict[node1]
        data_info2 = icd9_info_dict[node2]

        # 计算 L2 距离并存储
        differences = np.linalg.norm(data_info1 - data_info2, axis=1)
        average_difference = np.mean(differences)
        difference_dict[(node1, node2)] = average_difference
        difference_dict[(node2, node1)] = average_difference  # 对称性存储
        all_differences.append(average_difference)

# 转为 NumPy 数组，方便后续标准化
all_differences = np.array(all_differences)
max_differences = np.max(all_differences)
min_differences = np.min(all_differences)

# 标准化和计算相似性，构建边
for (node1, node2), difference in difference_dict.items():
    normed_difference = (difference - min_differences) / (max_differences - min_differences)
    similarity = 1 / (1 + normed_difference)  # 相似性

    G_u.add_edge(str(node1), str(node2), weight=similarity)

# 只保留前 50% 的边
edge_weights = [edge[2]['weight'] for edge in G_u.edges(data=True)]
remove_threshold = np.percentile(edge_weights, 50)
edges_to_remove = [(u, v) for u, v, weight in G_u.edges(data=True) if weight['weight'] < remove_threshold]
G_u.remove_edges_from(edges_to_remove)

# 保存图
with open(os.path.join(train_dir, f"usability_graph.pkl"), 'wb') as f:
    pickle.dump(G_u, f)

In [ ]:
# Visualize graph
with open(os.path.join(train_dir, "usability_graph.pkl"), 'rb') as f:
    G_u_target = pickle.load(f)

draw_graph(G_u_target, node_num=20)